# Phase 2 verification

This notebook verifies the local Phase 2 data contract without pytest, a live model, or network access. It covers P2-T1 through P2-T5 in `docs/acceptance.md`. P2-T6 remains pending until the official NHTSA/EPA adapters are implemented.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'car_agent').exists():
            return candidate
    raise RuntimeError('Could not find the car-agent repository root')

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / 'src'))
assert sys.version_info[:2] == (3, 12), f'Python 3.12 required, found {sys.version}'
print(f'Repository: {REPO_ROOT}')
print(f'Python: {sys.version.split()[0]}')

In [ ]:
from car_agent.data_pipeline import InventoryValidationError, load_inventory_fixture
from car_agent.repositories import KnowledgeRepository

results = {}

def fixture_record(**overrides):
    record = {
        'id': 'notebook-fixture-car',
        'make': 'Fixture',
        'model': 'Sport',
        'year': 2005,
        'price': 25000,
        'mileage': 50000,
        'body_style': 'coupe',
        'transmission': 'manual',
        'drivetrain': 'RWD',
        'horsepower': 250,
        'description': 'notebook fixture vehicle',
        'tags': ['weekend'],
        'provenance': {
            'source_url': 'local://phase2/notebook-fixture-car',
            'source_type': 'notebook_fixture',
            'retrieved_at': '2026-09-03T00:00:00Z',
        },
    }
    record.update(overrides)
    return record

## P2-T1 — normalize a valid source fixture

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    fixture_path = Path(directory) / 'valid.json'
    fixture_path.write_text(json.dumps([fixture_record()]))
    vehicles = load_inventory_fixture(fixture_path)

assert len(vehicles) == 1
assert 1990 <= vehicles[0].year <= 2020
assert vehicles[0].provenance.source_url == 'local://phase2/notebook-fixture-car'
results['P2-T1'] = 'PASS'
print(json.dumps(vehicles[0].to_dict(), indent=2))

## P2-T2 — reject missing or incomplete provenance

In [ ]:
invalid_provenance = [
    ('missing', {key: value for key, value in fixture_record().items() if key != 'provenance'}),
    ('incomplete', {**fixture_record(), 'provenance': {'source_type': 'notebook_fixture', 'retrieved_at': '2026-09-03T00:00:00Z'}}),
]
for label, record in invalid_provenance:
    with tempfile.TemporaryDirectory() as directory:
        fixture_path = Path(directory) / f'{label}.json'
        fixture_path.write_text(json.dumps([record]))
        try:
            load_inventory_fixture(fixture_path)
        except InventoryValidationError as exc:
            print(f'{label}: rejected as expected — {exc}')
        else:
            raise AssertionError(f'{label} provenance was accepted')
results['P2-T2'] = 'PASS'

## P2-T3 — load the same mock fixture twice deterministically

In [ ]:
first = load_inventory_fixture(REPO_ROOT / 'data' / 'inventory.json')
second = load_inventory_fixture(REPO_ROOT / 'data' / 'inventory.json')
assert [vehicle.id for vehicle in first] == [vehicle.id for vehicle in second]
assert len({vehicle.id for vehicle in first}) == len(first)
assert first == second
results['P2-T3'] = 'PASS'
print(f'Deterministic mock inventory records: {len(first)}')

## P2-T4 — retrieve model-specific sourced knowledge

In [ ]:
facts = KnowledgeRepository().retrieve('honda-s2000-2004', topic='ownership')
assert facts
assert all(fact.vehicle_id == 'honda-s2000-2004' for fact in facts)
assert all(fact.source for fact in facts)
results['P2-T4'] = 'PASS'
print(json.dumps([fact.to_dict() for fact in facts], indent=2))

## P2-T5 — reject invalid schema values

In [ ]:
invalid_cases = {
    'year': {'year': 1989},
    'price': {'price': 0},
    'mileage': {'mileage': -1},
    'id': {'id': ''},
}
for field, overrides in invalid_cases.items():
    with tempfile.TemporaryDirectory() as directory:
        fixture_path = Path(directory) / f'invalid-{field}.json'
        fixture_path.write_text(json.dumps([fixture_record(**overrides)]))
        try:
            load_inventory_fixture(fixture_path)
        except InventoryValidationError as exc:
            print(f'{field}: rejected as expected — {exc}')
        else:
            raise AssertionError(f'invalid {field} was accepted')
results['P2-T5'] = 'PASS'

## Project fixture audit and final result

In [ ]:
project_vehicles = load_inventory_fixture(REPO_ROOT / 'data' / 'inventory.json')
assert project_vehicles
assert all(vehicle.provenance and vehicle.provenance.source_url for vehicle in project_vehicles)
print(f'Project inventory records: {len(project_vehicles)}')
print('All project records have provenance.')
print('\nPhase 2 local acceptance results:')
for case_id, status in results.items():
    print(f'  [{status}] {case_id}')
assert set(results) == {'P2-T1', 'P2-T2', 'P2-T3', 'P2-T4', 'P2-T5'}
assert all(status == 'PASS' for status in results.values())
print('\nPhase 2 local verification: PASS')